# Reverse-mode automatic differentiation

Deep learning frameworks like PyTorch are built on **automatic differentiation** (autodiff). In the previous exercise, we computed gradients by hand and implemented them as Python functions. This works for a single neuron, but becomes impractical for models with thousands of parameters.

In this exercise, you will build a minimal autodiff engine from scratch. This will demystify how frameworks like PyTorch compute gradients under the hood.

In [ ]:
import numpy as np

## Background: the computational graph

Every mathematical expression can be broken down into a sequence of elementary operations (addition, multiplication, etc.), forming a **computational graph**. For example, the expression $d = a \cdot (a + b)$ becomes:

$$
c = a + b, \qquad d = a \cdot c
$$

To find $\frac{\partial d}{\partial a}$, we apply the chain rule along the paths from $d$ back to $a$ in the graph:

- Path 1: $d \to a$ directly (through the multiplication): $\frac{\partial d}{\partial a}\big|_{\text{direct}} = c$
- Path 2: $d \to c \to a$ (through the addition inside $c$): $\frac{\partial d}{\partial c} \cdot \frac{\partial c}{\partial a} = a \cdot 1 = a$

The total gradient is the **sum over all paths**: $\frac{\partial d}{\partial a} = c + a = (a + b) + a = 2a + b$.

**Reverse-mode autodiff** automates this process: it traverses the graph backward from the output, accumulating gradients along the way.

## Exercise 1: Verify by hand

For $a = 4$ and $b = 3$:

1. Compute $d = a \cdot (a + b)$ numerically.
2. Compute $\frac{\partial d}{\partial a}$ and $\frac{\partial d}{\partial b}$ using the formulas above.
3. Verify using a **finite difference approximation**. The idea: if we nudge $a$ by a tiny amount $\epsilon$, the ratio of how much $d$ changes to how much we nudged is approximately the derivative:  

$$ \frac{\partial d}{\partial a} \approx \frac{d(a + \epsilon, b) - d(a, b)}{\epsilon} $$

Compute this for both $\frac{\partial d}{\partial a}$ and $\frac{\partial d}{\partial b}$ with $\epsilon = 10^{-8}$ and compare to your analytical answers.

In [ ]:
# Your code here


## Exercise 2: Build an autodiff engine

We will implement a `Variable` class that automatically tracks the computational graph and computes gradients via reverse-mode autodiff.

The key idea: every `Variable` stores not just its value, but also a list of **(child, local_gradient)** pairs that record how it was computed. This forms the edges of the computational graph.

### a) Complete the implementation

The code below provides the `Variable` class and the `add` function. Your task is to:

1. **Complete `mul`**: fill in the value and local gradients for multiplication. Recall that $\frac{\partial}{\partial a}(a \cdot b) = b$ and $\frac{\partial}{\partial b}(a \cdot b) = a$.

2. **Complete `compute_gradients`**: this function traverses the graph backward. At each node, it:
   - *Multiplies* the incoming gradient by the local gradient (chain rule along an edge)
   - *Adds* the result to the total gradient for that child (summing over paths)

In [ ]:
from collections import defaultdict


class Variable:
    def __init__(self, value, gradients=None):
        self.value = value
        # If no gradients are given, this is a leaf variable.
        # Its gradient w.r.t. itself is 1 (well, sign(value) for technical reasons).
        self._gradients = (
            gradients if gradients is not None else ((self, np.sign(value)),)
        )
        self._stored_gradients = None

    @property
    def gradients(self):
        """Compute and cache the gradients of this variable w.r.t. all ancestors."""
        if self._stored_gradients is None:
            self._stored_gradients = dict(compute_gradients(self))
        return self._stored_gradients


def add(a, b):
    """Create the variable that results from adding two variables."""
    value = a.value + b.value
    gradients = (
        (a, 1),  # d/da (a + b) = 1
        (b, 1),  # d/db (a + b) = 1
    )
    return Variable(value, gradients)


def mul(a, b):
    """Create the variable that results from multiplying two variables."""
    value = ...  # TODO: compute a * b
    gradients = (
        (a, ...),  # TODO: d/da (a * b) = ?
        (b, ...),  # TODO: d/db (a * b) = ?
    )
    return Variable(value, gradients)


def compute_gradients(variable):
    """Compute the gradients of `variable` w.r.t. all leaf variables.

    Traverses the computational graph backward, accumulating gradients
    using the chain rule.
    """
    gradients = defaultdict(lambda: 0)

    def _compute_gradients(variable, total_gradient):
        for child_variable, local_gradient in variable._gradients:
            # "Multiply the edges of a path" (chain rule):
            gradient = ...  # TODO
            # "Add together the different paths":
            gradients[child_variable] = ...  # TODO

            # Stop recursion at leaf variables
            is_leaf = (
                len(child_variable._gradients) == 1
                and child_variable._gradients[0][0] is child_variable
            )
            if not is_leaf:
                _compute_gradients(child_variable, gradient)

    _compute_gradients(variable, total_gradient=1)
    return gradients

### b) Test your implementation

Use your `Variable` class to compute $d = a \cdot (a + b)$ with $a = 4$, $b = 3$, and verify that the gradient $\frac{\partial d}{\partial a} = 11$ and $\frac{\partial d}{\partial b} = 4$.

In [ ]:
a = Variable(4)
b = Variable(3)
# TODO: compute c = a + b, then d = a * c using add() and mul()
# Then check that d.gradients[a] == 11 and d.gradients[b] == 4

## Exercise 3: Autodiff for a neuron

Now let's use autodiff for something more interesting. We import a complete `Variable` class from `variable.py` (which includes additional operations like subtraction, division, exponentiation, and the `exp` function).

**Task:** Define a forward pass for a sigmoid neuron and verify that the autodiff gradient matches the analytical gradient.

The sigmoid neuron computes:

$$
\hat{y} = \sigma(w_0 + w_1 x), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}
$$

with loss $\mathcal{L} = \frac{1}{2}(y - \hat{y})^2$.

The analytical gradient of the loss w.r.t. $w_1$ is:

$$
\frac{\partial \mathcal{L}}{\partial w_1} = (\hat{y} - y) \cdot x \cdot \hat{y}(1 - \hat{y})
$$

Fill in the three functions below, then run the test.

In [ ]:
from variable import Variable, exp


def loss(y, y_hat):
    """Squared error loss."""
    return ...  # TODO


def sigma(z):
    """Sigmoid activation function."""
    return ...  # TODO


def y_hat(w_0, w_1, x_1):
    """Forward pass of a sigmoid neuron."""
    return ...  # TODO

In [ ]:
# --- Verification (just run this cell) ---

# Analytical gradient formulas
def dy_hat_dw_1(w_0, w_1, x_1):
    yh = y_hat(w_0, w_1, x_1)
    return x_1 * yh * (1 - yh)


def dloss_dw_1(y, w_0, w_1, x_1):
    return (y_hat(w_0, w_1, x_1) - y) * dy_hat_dw_1(w_0, w_1, x_1)


# Create variables
x_1 = Variable(0.1, name="x_1")
w_0 = Variable(4, name="w_0")
w_1 = Variable(3, name="w_1")
y = Variable(10, name="y")


def isclose(a, b):
    a = a if not isinstance(a, Variable) else a.value
    b = b if not isinstance(b, Variable) else b.value
    return np.isclose(a, b)


# Compare autodiff gradient with analytical gradient
autodiff_grad = loss(y, y_hat(w_0, w_1, x_1)).gradients[w_1]
analytical_grad = dloss_dw_1(y, w_0, w_1, x_1)

assert isclose(autodiff_grad, analytical_grad), (
    f"Mismatch: autodiff={autodiff_grad}, analytical={analytical_grad}"
)
print(f"Autodiff gradient:   {autodiff_grad}")
print(f"Analytical gradient: {analytical_grad}")
print("They match!")